# 05 · Model comparison & segment profiling

**Owner:** Person 6 — Model Comparison & Profiling  
**Run from:** the `notebooks/` folder

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `src` can be imported from notebooks/
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
FIG = Path.cwd().parent / "reports" / "figures"


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from src.preprocess import load_clean_data, FEATURES, RANDOM_STATE

df = load_clean_data()
X_pca = PCA(n_components=3, random_state=RANDOM_STATE).fit_transform(StandardScaler().fit_transform(df[FEATURES]))

## Compare three clustering models (k = 4)

In [ ]:
models = {
    "K-Means": KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE),
    "Agglomerative": AgglomerativeClustering(n_clusters=4),
    "Gaussian Mixture": GaussianMixture(n_components=4, random_state=RANDOM_STATE),
}
results, labels = [], {}
for name, m in models.items():
    labels[name] = m.fit_predict(X_pca)
    results.append({"model": name,
                    "silhouette ↑": silhouette_score(X_pca, labels[name]),
                    "davies_bouldin ↓": davies_bouldin_score(X_pca, labels[name]),
                    "calinski_harabasz ↑": calinski_harabasz_score(X_pca, labels[name])})
pd.DataFrame(results).set_index("model").round(3)

## Profile the final segments

In [ ]:
import joblib
from src.preprocess import MODEL_PATH

model = joblib.load(MODEL_PATH)   # created by notebook 04 / src/train.py
Xp = model["pca"].transform(model["scaler"].transform(df[FEATURES]))
df["Cluster"] = model["kmeans"].predict(Xp)
df["Segment"] = df["Cluster"].map(lambda c: model["segments"][c]["name"])
model["profile"]

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Income", y="Total_Spent", hue="Segment", alpha=0.6)
plt.title("Customer segments: income vs spend")
plt.savefig(FIG / "segments_income_spend.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, ["Income", "Total_Spent", "Age", "Children", "NumDealsPurchases", "NumWebVisitsMonth"]):
    sns.boxplot(data=df, x="Segment", y=col, ax=ax); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(FIG / "segment_boxplots.png", dpi=150); plt.show()

In [ ]:
from src.preprocess import MNT_COLS, PURCHASE_COLS
df.groupby("Segment")[MNT_COLS + PURCHASE_COLS].mean().round(1)

## Segment personas & marketing actions (TODO — refine)

| Segment | Who they are | Marketing action |
|---|---|---|
| Premium | | |
| Steady families | | |
| Budget parents | | |
| Young low spenders | | |